In [1]:
import csv, hashlib, itertools, json, statistics
from pathlib import Path
import transformers
from transformers import AutoTokenizer

RUN_ID = 'phase3_sro_token_audit_20261121_01'
BASE = Path('/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique')
RAW = BASE / 'fg_full_data'
TOK = BASE / 'fg_full_stage1/best_model_after_2_epochs'
SRO = Path('/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121')
OUT = Path('/kaggle/working') / RUN_ID
MAX_LENGTH = 512
EXPECTED = {
 'train': (46610, '7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D', 'A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8'),
 'dev': (6404, 'F7EC0B2308EBDC811BFC11828A076F498FD2CB542C503274DAD5CBA852EBD1BC', '2E469335EB7F58C20C9CA1225CB3E4B08338D92FC5B8B750D574C23FED85E302')}

def file_hash(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest().upper()

def pct(xs, p):
    a = sorted(xs)
    k = (len(a)-1)*p/100
    i = int(k)
    return a[i] + (a[min(i+1,len(a)-1)]-a[i])*(k-i)

def summary(xs):
    n = sum(x > MAX_LENGTH for x in xs)
    return {'mean': statistics.mean(xs), 'median': statistics.median(xs),
            'p90': pct(xs,90), 'p95': pct(xs,95), 'p99': pct(xs,99),
            'max': max(xs), 'over_512_rows': n, 'over_512_rate': n/len(xs)}

def passages(text):
    prefix, separator = 'Sub-question: ', '\n\nPassages:\n'
    if not text.startswith(prefix) or separator not in text:
        raise ValueError('invalid text contract')
    return text[len(prefix):].split(separator,1)[1]

def lengths(tok, texts):
    result=[]
    for i in range(0,len(texts),128):
        result.extend(tok(texts[i:i+128], add_special_tokens=True,
                          truncation=False, padding=False, return_length=True,
                          return_attention_mask=False, return_token_type_ids=False)['length'])
    return result

def audit(split,tok):
    expected_n, raw_hash, sro_hash = EXPECTED[split]
    raw_path, sro_path = RAW/f'deberta_fg_{split}.csv', SRO/f'{split}.csv'
    if file_hash(raw_path)!=raw_hash or file_hash(sro_path)!=sro_hash:
        raise ValueError(f'{split}: source or SRO hash mismatch')
    raw_texts=[]; sro_texts=[]; labels=[]
    with raw_path.open(encoding='utf-8-sig',newline='') as rf, sro_path.open(encoding='utf-8-sig',newline='') as sf:
        rr,sr=csv.DictReader(rf),csv.DictReader(sf)
        if rr.fieldnames!=['text','label'] or sr.fieldnames!=['text','label']:
            raise ValueError(f'{split}: unexpected CSV header')
        for i,(r,s) in enumerate(itertools.zip_longest(rr,sr)):
            if r is None or s is None or r['label']!=s['label'] or r['label'] not in ('0','1'):
                raise ValueError(f'{split}: row or label mismatch at {i}')
            if passages(r['text']) != passages(s['text']):
                raise ValueError(f'{split}: passage mismatch at {i}')
            raw_texts.append(r['text']); sro_texts.append(s['text']); labels.append(r['label'])
    if len(labels)!=expected_n:
        raise ValueError(f'{split}: expected {expected_n}, got {len(labels)}')
    a,b=lengths(tok,raw_texts),lengths(tok,sro_texts)
    out_csv=OUT/f'{split}_paired_token_lengths.csv'
    with out_csv.open('x',encoding='utf-8',newline='') as f:
        w=csv.writer(f); w.writerow(['row_index','label','raw_tokens','sro_tokens','raw_over_512','sro_over_512'])
        for i,(y,x,z) in enumerate(zip(labels,a,b)):
            w.writerow([i,y,x,z,int(x>MAX_LENGTH),int(z>MAX_LENGTH)])
    delta=[z-x for x,z in zip(a,b)]
    transitions={'both_within_512':sum(x<=512 and z<=512 for x,z in zip(a,b)),
                 'raw_only_over_512':sum(x>512 and z<=512 for x,z in zip(a,b)),
                 'sro_only_over_512':sum(x<=512 and z>512 for x,z in zip(a,b)),
                 'both_over_512':sum(x>512 and z>512 for x,z in zip(a,b))}
    return {'rows':len(labels),'raw_sha256':raw_hash,'sro_sha256':sro_hash,
            'paired_passages_labels_order':'passed','raw':summary(a),'sro':summary(b),
            'mean_delta_sro_minus_raw':statistics.mean(delta),
            'median_delta_sro_minus_raw':statistics.median(delta),
            'over_512_transitions':transitions,'paired_lengths_sha256':file_hash(out_csv)}

if OUT.exists():
    raise FileExistsError(OUT)
for p in [TOK/'tokenizer.json', TOK/'tokenizer_config.json',
          RAW/'deberta_fg_train.csv',RAW/'deberta_fg_dev.csv',SRO/'train.csv',SRO/'dev.csv']:
    if not p.is_file(): raise FileNotFoundError(p)
tok=AutoTokenizer.from_pretrained(TOK,local_files_only=True,use_fast=True)
if not tok.is_fast: raise ValueError('expected checkpoint fast tokenizer')
OUT.mkdir(parents=True)
report={'run_id':RUN_ID,'tokenizer_source':str(TOK),
        'tokenizer_sha256':{n:file_hash(TOK/n) for n in ['tokenizer.json','tokenizer_config.json']},
        'transformers_version':transformers.__version__,'max_length':MAX_LENGTH,
        'tokenization':'full CSV text; add_special_tokens=True; truncation=False; padding=False',
        'splits':{s:audit(s,tok) for s in ('train','dev')}}
with (OUT/'token_length_audit.json').open('x',encoding='utf-8') as f:
    json.dump(report,f,ensure_ascii=False,indent=2); f.write('\n')
print(json.dumps(report,ensure_ascii=False,indent=2))

{
  "run_id": "phase3_sro_token_audit_20261121_01",
  "tokenizer_source": "/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_stage1/best_model_after_2_epochs",
  "tokenizer_sha256": {
    "tokenizer.json": "A29E21FAEC7433EE508590C7BEE5DC869FDDB335AACF11D53BE766CF0FD81294",
    "tokenizer_config.json": "0DBE41323167BA5F2E1E876CCD05068318DD2378B922BDFD924A389DBA3BCC7A"
  },
  "transformers_version": "5.20.2",
  "max_length": 512,
  "tokenization": "full CSV text; add_special_tokens=True; truncation=False; padding=False",
  "splits": {
    "train": {
      "rows": 46610,
      "raw_sha256": "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D",
      "sro_sha256": "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8",
      "paired_passages_labels_order": "passed",
      "raw": {
        "mean": 330.6868483158121,
        "median": 335.0,
        "p90": 372.0,
        "p95": 387.0,
        "p99": 418.0,
        "max": 505,
     